In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = Path.cwd().parent   # nbconvert and Jupyter both start the kernel in notebooks/
assert (ROOT / "paths.py").is_file(), f"paths.py not found from {Path.cwd()}"
sys.path.insert(0, str(ROOT))

# Step 5, second learner: FLANN-style ridge regression and map-safe variants

Same target, folds, fold baselines, training rows and benchmark as `residual_lobo.ipynb`. The learner is ridge regression on a fixed functional-link basis (`residual.make_flann_basis`), set in `docs/plan.md` before the first run:

- daily harmonics 1-3 of LTST and seasonal harmonics 1-2 of Ls, each times (1, tau), and tau;
- linear static terms (TI, slope, sin/cos aspect, albedo) and the baseline, each times (1, sin, cos of the first daily harmonic);
- no elevation; standardized columns; ridge alpha 1.0, fixed.

## Variants (configurations 5-8)

| Variant | Albedo terms | Clamped | Status |
|---|---|---|---|
| `flann` | yes | no | configuration 5 (first run): adopted over `no_elev`, but albedo extrapolates by 29 K on the map |
| `flann_clamped` | yes | yes | configuration 6, attribution only |
| `flann_no_albedo` | no | no | configuration 7, attribution only |
| `flann_safe` | no | yes | configuration 8, the map candidate |
| `flann_safe_no_ti` | no (and no TI terms) | yes | configuration 9 (added by user request): TI sensitivity only, never selected |

**Clamping:** TI, slope, albedo and the baseline (as a predictor only) are clipped to the fold's training-row 1st-99th percentiles, in training and prediction. The residual is still added to the unclamped baseline. This is what the map pipeline will do, so its LOBO score is the map's score.

## Decision rule (fixed before running)

1. Configuration 5: adopt `flann` over `no_elev` if the macro RMSE gain is at least 0.2 K and it is better in at least 5 of 9 blocks.
2. Configuration 8: `flann_safe` becomes the chosen model if it passes the same test against `no_elev`, even if `flann` scores better (the reason for it is map safety). If it fails, the chosen model is `no_elev`, and `flann` is not used on the map.

Fold 0 is reported on its own: block 0's TI (566-682 tiu) lies above every other block's (at most 514 tiu). Clamping removes FLANN's TI extrapolation there by design.

In [ ]:
# --- config ---
import time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import paths
import thermal1d as t
import residual as r

SUSPECT_SOLS = (3763, 3778)
EXPECTED_TRAIN_BLOCKS = [0, 1, 2, 3, 5, 6, 7, 9, 10]
LS_BIN_DEG = 30
ADOPT_MIN_GAIN_K, ADOPT_MIN_BLOCK_WINS = 0.2, 5
MAP_CANDIDATE = "flann_safe"
REPRO_TOL_K = 0.05
NIGHT_HOURS = [0, 1, 2, 3, 4, 5, 19, 20, 21, 22, 23]
EXTENDED = {"ti_themis_tiu": (300.0, 750.0), "slope_deg": (0.0, 30.0), "albedo_mean": (0.10, 0.35)}
FIT_PARAMS = paths.OUT_DIR / "fit_params.csv"
STEP5_ROWS = paths.RESIDUAL_OUT_DIR / "residual_rows.csv"
OUT = paths.RESIDUAL_OUT_DIR
VARIANTS = r.FLANN_VARIANTS
print("FLANN alpha:", r.FLANN_RIDGE_ALPHA, "| clamp:", r.CLAMP_COLUMNS, r.CLAMP_PERCENTILES)
print("variants:", {k: (len(v[0]), v[1]) for k, v in VARIANTS.items()})

## Data and fold baselines (as in `residual_lobo.ipynb`)

In [ ]:
tbl = pd.DataFrame(t.load_rems_table(paths.DATA_DIR / "gale_rems_training.csv"))
assert (tbl.split_role == "train").all()
blocks = sorted(tbl.block.unique())
assert blocks == EXPECTED_TRAIN_BLOCKS, f"training blocks {blocks}"
per_sol_cols = ["ls_deg", "tau", "tau_is_placeholder", "row", "slope_deg", "aspect_deg",
                "albedo_mean", "ti_themis_tiu", "block"]
assert (tbl.groupby("sol")[per_sol_cols].nunique() == 1).all().all(), "per-sol inputs vary within a sol"
sols = tbl.groupby("sol")[per_sol_cols].first().reset_index()
sol_inputs = {"lat_deg": t.lat_from_row(sols.row.to_numpy()), "ls_deg": sols.ls_deg.to_numpy(),
              "tau": sols.tau.to_numpy(), "slope_deg": sols.slope_deg.to_numpy(),
              "aspect_deg": sols.aspect_deg.to_numpy(), "albedo": sols.albedo_mean.to_numpy(),
              "ti": sols.ti_themis_tiu.to_numpy()}
row_sol = pd.Series(np.arange(len(sols)), index=sols.sol)[tbl.sol].to_numpy()
row_step = t.bin_centre_step(tbl.bin.to_numpy())
evaluate = t.make_evaluator(sol_inputs, row_sol, row_step)

obs = tbl.ground_temp_k.to_numpy()
block = tbl.block.to_numpy()
suspect = tbl.sol.between(*SUSPECT_SOLS).to_numpy()
placeholder = tbl.tau_is_placeholder.to_numpy()
fit_ok = ~suspect & ~placeholder
tbl["group"] = np.where(suspect, "suspect_terrain", np.where(placeholder, "placeholder_tau", "main"))
tbl["ltst_hour"] = (tbl.bin // 4).astype(int)
tbl["ltst_hr"] = (tbl.bin + 0.5) * t.BIN_HOURS
tbl["ls_bin"] = (tbl.ls_deg // LS_BIN_DEG * LS_BIN_DEG).astype(int)
tbl["q_abs_wm2"] = 0.0     # not in the FLANN basis; make_features needs the column

step5 = pd.read_csv(STEP5_ROWS)
assert (step5.sol.to_numpy() == tbl.sol.to_numpy()).all() and (step5.bin.to_numpy() == tbl.bin.to_numpy()).all(), \
    "row order differs from Step 5"
fits = pd.read_csv(FIT_PARAMS).set_index("fold")
fold_base = {}
t0 = time.time()
for label in fits.index:
    p = tuple(fits.loc[label, list(t.PARAM_NAMES)].to_numpy(dtype=float))
    fold_base[label] = evaluate(p).copy()
    assert np.all(np.isfinite(fold_base[label])), f"non-finite baseline in {label}"
print(f"fold baselines in {time.time() - t0:.0f} s")
oof = np.full(len(tbl), np.nan)
for b in blocks:
    oof[block == b] = fold_base[f"block {b}"][block == b]
diff = np.abs(oof - step5.baseline_k.to_numpy()).max()
print(f"max |fold baseline - Step 5 baseline_k| = {diff:.4f} K")
assert diff < REPRO_TOL_K, "fold baselines do not reproduce Step 5"
tbl["baseline_k"] = oof
tbl["bench_k"] = step5.bench_k.to_numpy()
tbl["pred_no_elev"] = step5.pred_no_elev.to_numpy()

## LOBO with each FLANN variant

In [ ]:
class FlannModel:
    """A fitted FLANN variant: clamp ranges (or None), static terms and the ridge pipeline."""

    def __init__(self, static, clamp, frame, target):
        self.static = static
        self.ranges = r.clamp_ranges(frame) if clamp else None
        self.pipe = r.make_flann_learner().fit(self.basis(frame)[0], target)

    def basis(self, frame):
        frac = {}
        if self.ranges is not None:
            frame, frac = r.apply_clamp(frame, self.ranges)
        return r.make_flann_basis(frame, self.static), frac

    def predict(self, frame):
        B, frac = self.basis(frame)
        return self.pipe.predict(B), frac


models = {v: {} for v in VARIANTS}
clip_log = []
for v in VARIANTS:
    tbl[f"pred_{v}"] = np.nan
for b in blocks + ["all"]:
    label = f"block {b}" if b != "all" else "all training blocks"
    held = block == b if b != "all" else np.zeros(len(tbl), bool)
    train = fit_ok & ~held
    frame = tbl.assign(baseline_k=fold_base[label])
    target = obs - fold_base[label]
    for v, (static, clamp) in VARIANTS.items():
        m = FlannModel(static, clamp, frame[train], target[train])
        models[v][b] = m
        if held.any():
            pred, frac = m.predict(frame[held])
            tbl.loc[held, f"pred_{v}"] = fold_base[label][held] + pred
            if clamp and v == MAP_CANDIDATE:
                clip_log.append({"block": b, **{f"clipped_{c}": f for c, f in frac.items()}})
assert tbl.filter(like="pred_flann").notna().all().all()
clip_log = pd.DataFrame(clip_log).set_index("block")
print("Held-out rows clipped by the fold's clamp ranges (fraction):")
print(clip_log.round(3).to_string())

## Scores and the decision rule

In [ ]:
MODELS = ["baseline", "bench", "no_elev"] + list(VARIANTS)
COLS = {"baseline": "baseline_k", "bench": "bench_k", **{m: f"pred_{m}" for m in MODELS[2:]}}
for m in MODELS:
    tbl[f"resid_{m}"] = tbl[COLS[m]] - tbl.ground_temp_k


def score(g, models=MODELS):
    out = {"n": len(g)}
    for m in models:
        out[f"{m}_rmse_k"] = np.sqrt((g[f"resid_{m}"] ** 2).mean())
        out[f"{m}_bias_k"] = g[f"resid_{m}"].mean()
    return pd.Series(out)


scored = tbl[tbl.group != "placeholder_tau"]
main = tbl[tbl.group == "main"]
by_block = scored.groupby("block").apply(score, include_groups=False)
by_group = tbl.groupby("group").apply(score, include_groups=False)
summary = pd.DataFrame({
    "macro_rmse_k": {m: by_block[f"{m}_rmse_k"].mean() for m in MODELS},
    "macro_rmse_no_3763_k": {m: by_block.drop(index=10)[f"{m}_rmse_k"].mean() for m in MODELS},
    "macro_rmse_no_block0_k": {m: by_block.drop(index=0)[f"{m}_rmse_k"].mean() for m in MODELS},
    "pooled_rmse_main_k": {m: np.sqrt((main[f"resid_{m}"] ** 2).mean()) for m in MODELS},
    "rmse_placeholder_tau_k": {m: by_group.loc["placeholder_tau", f"{m}_rmse_k"] for m in MODELS},
    "rmse_suspect_terrain_k": {m: by_group.loc["suspect_terrain", f"{m}_rmse_k"] for m in MODELS},
    "blocks_beating_bench": {m: int((by_block[f"{m}_rmse_k"] < by_block.bench_rmse_k).sum()) for m in MODELS},
    "blocks_beating_no_elev": {m: int((by_block[f"{m}_rmse_k"] < by_block.no_elev_rmse_k).sum()) for m in MODELS},
})
pd.options.display.float_format = "{:.2f}".format
print(summary.to_string())
print()
print("Per-block RMSE (K):")
print(by_block[["n"] + [f"{m}_rmse_k" for m in MODELS]].to_string())
print()
print("Per-block bias (K):")
print(by_block[[f"{m}_bias_k" for m in MODELS]].to_string())


def passes(m):
    gain = summary.macro_rmse_k["no_elev"] - summary.macro_rmse_k[m]
    wins = summary.blocks_beating_no_elev[m]
    ok = gain >= ADOPT_MIN_GAIN_K and wins >= ADOPT_MIN_BLOCK_WINS
    return ok, f"{m} vs no_elev: macro gain {gain:+.3f} K (need >= {ADOPT_MIN_GAIN_K}), better in {wins}/9 blocks (need >= {ADOPT_MIN_BLOCK_WINS})"


ok5, line5 = passes("flann")
ok8, line8 = passes(MAP_CANDIDATE)
chosen = MAP_CANDIDATE if ok8 else "no_elev"
log = [f"5. {line5} -> {'passes' if ok5 else 'fails'}",
       f"8. {line8} -> {'passes' if ok8 else 'fails'}",
       f"chosen residual model for the map: {chosen}"]
print()
print(chr(10).join(log))
(OUT / "flann_decision.txt").write_text(chr(10).join(log) + chr(10))

In [ ]:
show = ["baseline", "bench", "no_elev", "flann", MAP_CANDIDATE]
b0 = tbl[tbl.block == 0]
night0 = b0[b0.ltst_hour.isin(NIGHT_HOURS)]
day0 = b0[~b0.ltst_hour.isin(NIGHT_HOURS)]
fold0 = pd.DataFrame({m: {"rmse_k": np.sqrt((b0[f"resid_{m}"] ** 2).mean()), "bias_k": b0[f"resid_{m}"].mean(),
                          "night_bias_k": night0[f"resid_{m}"].mean(), "day_bias_k": day0[f"resid_{m}"].mean()}
                      for m in MODELS}).T
print("Fold 0, all block 0 rows (its TI is outside the training range):")
print(fold0.to_string())

by_hour = main.groupby("ltst_hour").apply(score, models=show, include_groups=False)
by_ls = main.groupby("ls_bin").apply(score, models=show, include_groups=False)
print()
print("By LTST hour (main rows):")
print(by_hour[[f"{m}_bias_k" for m in show] + [f"{m}_rmse_k" for m in show]].to_string())
print()
print("By Ls bin (main rows):")
print(by_ls[["n"] + [f"{m}_bias_k" for m in show]].to_string())

## Covariate effects by hour, inside and beyond the rover's range

For each static covariate, the mean predicted residual per LTST hour as the covariate is swept, with other inputs at the training rows (all-blocks models). Effect = the largest spread over the sweep at any hour. "Rover range" is the training rows' 1st-99th percentile; "extended" is a range the map will reach. Clamped variants apply their clamp, as the map would.

In [ ]:
def hourly_effect(m, label, rows, col, values):
    frame = tbl[rows].assign(baseline_k=fold_base[label][rows])
    hour = frame.ltst_hour.to_numpy()
    return pd.DataFrame([pd.Series(m.predict(frame.assign(**{col: v}))[0]).groupby(hour).mean() for v in values],
                        index=values)


eff = []
for v in VARIANTS:
    for col, (elo, ehi) in EXTENDED.items():
        lo, hi = np.percentile(tbl.loc[fit_ok, col], [1, 99])
        for rng_name, (a, b) in (("rover range", (lo, hi)), ("extended", (elo, ehi))):
            h = hourly_effect(models[v]["all"], "all training blocks", fit_ok, col, np.linspace(a, b, 11))
            spread = h.max() - h.min()
            eff.append({"variant": v, "covariate": col, "range": rng_name, "lo": a, "hi": b,
                        "max_spread_k": spread.max(), "at_hour": int(spread.idxmax())})
eff = pd.DataFrame(eff)
eff.to_csv(OUT / "flann_covariate_effects.csv", index=False)
print("Largest hourly spread of the predicted residual (K), all-blocks models:")
print(eff.pivot_table(index=["covariate", "range"], columns="variant", values="max_spread_k").to_string())

print()
print("Fold-0 models (trained without block 0): mean predicted residual at TI 450 / 514 / 620 tiu")
for v in VARIANTS:
    h0 = hourly_effect(models[v][0], "block 0", fit_ok & (block != 0), "ti_themis_tiu", np.array([450.0, 514.0, 620.0]))
    night = h0[[c for c in h0.columns if c in NIGHT_HOURS]].mean(axis=1).to_numpy()
    day = h0[[c for c in h0.columns if c not in NIGHT_HOURS]].mean(axis=1).to_numpy()
    print(f"  {v:16s} night {np.round(night, 2)} K | day {np.round(day, 2)} K")

cand = models[MAP_CANDIDATE]["all"]
coef = pd.Series(cand.pipe[-1].coef_, index=cand.basis(tbl[fit_ok].head(1).assign(baseline_k=250.0))[0].columns)
print()
print(f"{MAP_CANDIDATE}, all-blocks standardized coefficients of the static terms (K per standard deviation):")
print(coef[[c for c in coef.index if any(c.startswith(s) for s in r.FLANN_STATIC)]].to_string())
print("all-blocks clamp ranges:", {k: tuple(round(float(x), 3) for x in v) for k, v in cand.ranges.items()})

In [ ]:
keep_cols = ["sol", "bin", "block", "group", "ground_temp_k", "baseline_k", "bench_k", "pred_no_elev"] + \
    [f"pred_{v}" for v in VARIANTS]
tbl[keep_cols].to_csv(OUT / "flann_rows.csv", index=False)
summary.to_csv(OUT / "flann_summary.csv")
by_block.to_csv(OUT / "flann_by_block.csv")
by_hour.to_csv(OUT / "flann_by_ltst_hour.csv")
by_ls.to_csv(OUT / "flann_by_ls_bin.csv")
fold0.to_csv(OUT / "flann_fold0.csv")
clip_log.to_csv(OUT / "flann_clipped_fraction.csv")
pd.DataFrame(cand.ranges, index=["lo", "hi"]).to_csv(OUT / "flann_safe_clamp_ranges.csv")

bars = ["bench", "no_elev", "flann", MAP_CANDIDATE]
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
ax = axes[0]
x = np.arange(len(by_block))
for i, m in enumerate(bars):
    ax.bar(x + (i - 1.5) * 0.2, by_block[f"{m}_rmse_k"], 0.2, label=m)
ax.set_xticks(x, [str(b) for b in by_block.index])
ax.set(xlabel="held-out block", ylabel="RMSE (K)", title="Leave-one-block-out RMSE")
ax.legend(fontsize=8)
ax = axes[1]
for m, st in (("baseline", "r-"), ("bench", "b--"), ("no_elev", "g-"), ("flann", "m:"), (MAP_CANDIDATE, "m-")):
    ax.plot(by_hour.index, by_hour[f"{m}_bias_k"], st, label=m)
ax.axhline(0, color="grey", lw=0.8)
ax.set(xlabel="LTST hour", ylabel="bias (K)", title="Bias by LTST hour (main rows)")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT / "flann_scores.png", dpi=120)
print("saved to", OUT)

## TI sensitivity across the crater (estimate, no solver run)

How much the map would change if the residual's TI terms were dropped (`flann_safe` vs `flann_safe_no_ti`, all-blocks models). For each TI value, D(TI, hour) = mean over the training rows of the difference between the two residuals, with that TI substituted (and clamped, as the map would). Each crater pixel's THEMIS TI (100 m feature table, TI column only) is then looked up in D. Slope, aspect, the baseline and the time inputs stay at the rover's distribution, so this is an estimate of the TI part only, not a map.

In [ ]:
FEATURES = paths.DATA_DIR / "gale_dust_devil_features_100m.csv"
DIFF_THRESHOLDS_K = (2.0, 5.0)
crater_ti = pd.read_csv(FEATURES, usecols=["ti_themis_tiu"]).ti_themis_tiu.to_numpy()
valid = np.isfinite(crater_ti)
print(f"{valid.sum()} of {crater_ti.size} pixels have THEMIS TI")
crater_ti = crater_ti[valid]
pct = np.percentile(crater_ti, [1, 5, 25, 50, 75, 95, 99])
print("crater THEMIS TI percentiles 1/5/25/50/75/95/99:", np.round(pct, 0))

safe, no_ti = models["flann_safe"]["all"], models["flann_safe_no_ti"]["all"]
lo_ti, hi_ti = safe.ranges["ti_themis_tiu"]
print(f"clamp range {lo_ti:.0f}-{hi_ti:.0f} tiu: {np.mean(crater_ti < lo_ti):.1%} of pixels below, "
      f"{np.mean(crater_ti > hi_ti):.1%} above (held at the edge)")

frame = tbl[fit_ok].assign(baseline_k=fold_base["all training blocks"][fit_ok])
hour = frame.ltst_hour.to_numpy()
base_no_ti = pd.Series(no_ti.predict(frame)[0]).groupby(hour).mean()
ti_grid = np.linspace(lo_ti, hi_ti, 33)
D = pd.DataFrame([pd.Series(safe.predict(frame.assign(ti_themis_tiu=v))[0]).groupby(hour).mean() - base_no_ti
                  for v in ti_grid], index=ti_grid)
night_cols = [c for c in D.columns if c in NIGHT_HOURS]
D_night = D[night_cols].mean(axis=1).to_numpy()
D_maxabs = D.abs().max(axis=1).to_numpy()
ti_c = np.clip(crater_ti, lo_ti, hi_ti)
pix_night = np.interp(ti_c, ti_grid, D_night)
pix_maxabs = np.interp(ti_c, ti_grid, D_maxabs)
crater = {"night_diff_pct_1_50_99_k": np.round(np.percentile(pix_night, [1, 50, 99]), 2).tolist(),
          **{f"frac_night_abs_diff_gt_{k:g}k": float(np.mean(np.abs(pix_night) > k)) for k in DIFF_THRESHOLDS_K},
          **{f"frac_any_hour_abs_diff_gt_{k:g}k": float(np.mean(pix_maxabs > k)) for k in DIFF_THRESHOLDS_K}}
print()
print("flann_safe minus flann_safe_no_ti (K), by TI (night mean and largest |difference| at any hour):")
print(pd.DataFrame({"ti_tiu": ti_grid, "night_mean_k": D_night, "max_abs_any_hour_k": D_maxabs}).iloc[::4].to_string(index=False))
print()
print("Over crater pixels:", crater)
rover_ti = tbl.loc[fit_ok, "ti_themis_tiu"]
print(f"for reference, rover rows: TI median {rover_ti.median():.0f} tiu; "
      f"night difference at that TI {np.interp(rover_ti.median(), ti_grid, D_night):+.2f} K")
D.to_csv(OUT / "ti_terms_difference_by_ti_hour.csv")
pd.Series(crater).to_csv(OUT / "ti_terms_crater_summary.csv")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ax = axes[0]
ax.hist(crater_ti, bins=80, color="grey")
ax.axvspan(lo_ti, hi_ti, color="m", alpha=0.15, label="clamp range (rover)")
ax.set(xlabel="THEMIS TI (tiu)", ylabel="pixels", title="Crater TI vs the rover's range")
ax.legend(fontsize=8)
ax = axes[1]
ax.plot(ti_grid, D_night, "m-", label="night mean (19-06 h)")
ax.plot(ti_grid, D_maxabs, "k--", label="largest |difference|, any hour")
ax.axhline(0, color="grey", lw=0.8)
ax.set(xlabel="THEMIS TI (tiu, clamped)", ylabel="K", title="flann_safe minus flann_safe_no_ti")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT / "ti_terms_crater_sensitivity.png", dpi=120)